<a href="https://colab.research.google.com/github/maxcyberkiller2000/domashkaSkillSpace/blob/main/%D0%A0%D0%B0%D1%81%D0%BF%D0%BE%D0%B7%D0%BD%D0%BE%D0%B2%D0%B0%D0%BD%D0%B8%D0%B5_%D0%BB%D0%B8%D1%86.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
import cv2
import numpy as np
from PIL import Image, ImageDraw
import requests
from io import BytesIO

def blur_face(image, factor=3):
    """Размытие изображения с заданным коэффициентом"""
    h, w = image.shape[:2]

    # Уменьшаем изображение и возвращаем к исходному размеру для размытия
    kernel_w = int(w / factor)
    kernel_h = int(h / factor)

    if kernel_w % 2 == 0:
        kernel_w -= 1
    if kernel_h % 2 == 0:
        kernel_h -= 1

    # Применяем размытие по Гауссу
    blurred = cv2.GaussianBlur(image, (kernel_w, kernel_h), 0)
    return blurred

def detect_and_process_face(image_path, output_path):
    # Загружаем изображение
    image = cv2.imread(image_path)


    # Конвертируем в RGB (OpenCV использует BGR)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original = image_rgb.copy()

    # Инициализируем детектор лиц
    face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    eye_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')

    # Обнаруживаем лица
    gray = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 4)

    if len(faces) == 0:
        print("Лица не обнаружены")
        return

    for (x, y, w, h) in faces:
        # Рисуем овал вокруг лица
        center_x = x + w // 2
        center_y = y + h // 2
        axes_x = w // 2
        axes_y = h // 2

        # Создаем маску для овала
        mask = np.zeros_like(image_rgb)
        cv2.ellipse(mask, (center_x, center_y), (axes_x, axes_y), 0, 0, 360, (255, 255, 255), -1)

        # Область лица
        face_region = image_rgb[y:y+h, x:x+w]

        # размываем всю область лица
        blurred_face = blur_face(face_region, factor=8)

        # Область для обнаружения глаз
        roi_gray = gray[y:y+h, x:x+w]
        roi_color = image_rgb[y:y+h, x:x+w]

        # Обнаруживаем глаза
        eyes = eye_cascade.detectMultiScale(roi_gray, 1.1, 4)

        # Создаем маску для глаз
        eyes_mask = np.zeros_like(roi_color)

        for (ex, ey, ew, eh) in eyes:
            # Рисуем круги вокруг глаз
            eye_center = (ex + ew//2, ey + eh//2)
            radius = max(ew, eh) // 2
            cv2.circle(eyes_mask, eye_center, radius, (255, 255, 255), -1)

            # также рисуем круги на исходном изображении для визуализации
            cv2.circle(original[y:y+h, x:x+w], eye_center, radius, (255, 0, 0), 2)

        # Объединяем размытое лицо с непокрытыми глазами
        eyes_region = cv2.bitwise_and(roi_color, eyes_mask)
        blurred_without_eyes = cv2.bitwise_and(blurred_face, cv2.bitwise_not(eyes_mask))
        final_face = cv2.bitwise_or(blurred_without_eyes, eyes_region)

        # Заменяем область лица на обработанную версию
        image_rgb[y:y+h, x:x+w] = final_face

        # Рисуем овал вокруг лица на финальном изображении
        cv2.ellipse(image_rgb, (center_x, center_y), (axes_x, axes_y), 0, 0, 360, (0, 255, 0), 2)

    # Конвертируем обратно в BGR для сохранения
    result_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    # Сохраняем результат
    cv2.imwrite(output_path, result_bgr)
    print(f"Результат сохранен в: {output_path}")

    # Показываем оригинал и результат
    cv2.imshow('Original', cv2.cvtColor(original, cv2.COLOR_RGB2BGR))
    cv2.imshow('Processed', result_bgr)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

# Использование функции
if __name__ == "__main__":
    # Замените путь на ваше изображение
    input_image = "actor_face.jpg"  # указываем путь для фотографии актера или актрисы
    output_image = "result_face.jpg"

    detect_and_process_face(input_image, output_image)

Результат сохранен в: processed_face.jpg


DisabledFunctionError: cv2.imshow() is disabled in Colab, because it causes Jupyter sessions
to crash; see https://github.com/jupyter/notebook/issues/3935.
As a substitution, consider using
  from google.colab.patches import cv2_imshow
